# Notebook 7 - Recherche d'hyperparametres du Transformer enrichi

Ce notebook reprend le protocole du notebook 6, mais ajoute une vraie recherche d'hyperparametres. L'objectif est de tester plusieurs configurations de fine-tuning du Transformer enrichi, puis de sauvegarder la meilleure configuration pour Streamlit.

## Difference entre optimisateur et recherche d'hyperparametres

- **AdamW** optimise les poids internes du Transformer pendant un entrainement donne.
- **La recherche d'hyperparametres** relance plusieurs entrainements avec des valeurs differentes : learning rate, batch size, longueur maximale, nombre d'epochs, seuils, etc.

Ici, chaque candidat est entraine avec AdamW via le `Trainer` Hugging Face, puis compare sur le jeu de validation.

In [1]:
from pathlib import Path
import gc
import os
import sys
import time

PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent
elif not (PROJECT_DIR / 'src').exists() and (PROJECT_DIR.parent / 'src').exists():
    PROJECT_DIR = PROJECT_DIR.parent

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import joblib
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import classification_report, f1_score, hamming_loss, jaccard_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

from src.baseline_ml import apply_thresholds, find_best_global_threshold, find_best_label_thresholds
from src.enriched_dataset import build_enriched_dataset, load_original_dataset, summarize_labels
from src.project_config import ENRICHED_GENRE_VOCABULARY
from src.transformer_model import NovelForgeTransformer, TransformerConfig

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 140)

DEVICE_KIND = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device available: {DEVICE_KIND}')
PROJECT_DIR

Device available: cpu


WindowsPath('C:/Users/ClémentPERRET/OneDrive - EQUATERRE-VDS/Bureau/Cours/DeepLearning')

## Parametres de recherche

La grille reste volontairement limitee sur CPU. Sur GPU, on augmente le nombre de lignes et on teste davantage de candidats.

In [2]:
RANDOM_STATE = 42
MODEL_NAME = 'distilbert-base-uncased'
SEARCH_METRIC = 'valid_f1_micro_global'

# if DEVICE_KIND == 'cuda':
CURRENT_TRAIN_ROWS = 8_000
MAL_TRAIN_ROWS = 12_000
VALID_ROWS = 2_000
TEST_ROWS = 3_000
HYPERPARAMETER_GRID = [
    {'learning_rate': 2e-5, 'batch_size': 16, 'epochs': 3, 'max_length': 192, 'weight_decay': 0.01},
    {'learning_rate': 3e-5, 'batch_size': 16, 'epochs': 3, 'max_length': 192, 'weight_decay': 0.01},
    {'learning_rate': 2e-5, 'batch_size': 16, 'epochs': 4, 'max_length': 256, 'weight_decay': 0.01},
    {'learning_rate': 1e-5, 'batch_size': 16, 'epochs': 4, 'max_length': 256, 'weight_decay': 0.02},
    {'learning_rate': 3e-5, 'batch_size': 32, 'epochs': 3, 'max_length': 192, 'weight_decay': 0.01},
]
# else:
#     CURRENT_TRAIN_ROWS = 1_500
#     MAL_TRAIN_ROWS = 2_500
#     VALID_ROWS = 600
#     TEST_ROWS = 900
#     HYPERPARAMETER_GRID = [
#         {'learning_rate': 2e-5, 'batch_size': 8, 'epochs': 2, 'max_length': 160, 'weight_decay': 0.01},
#         {'learning_rate': 3e-5, 'batch_size': 8, 'epochs': 2, 'max_length': 192, 'weight_decay': 0.01},
#         {'learning_rate': 1e-5, 'batch_size': 8, 'epochs': 3, 'max_length': 192, 'weight_decay': 0.02},
#     ]

print({
    'current_train_rows': CURRENT_TRAIN_ROWS,
    'mal_train_rows': MAL_TRAIN_ROWS,
    'valid_rows': VALID_ROWS,
    'test_rows': TEST_ROWS,
    'candidates': len(HYPERPARAMETER_GRID),
    'search_metric': SEARCH_METRIC,
})
pd.DataFrame(HYPERPARAMETER_GRID)

{'current_train_rows': 8000, 'mal_train_rows': 12000, 'valid_rows': 2000, 'test_rows': 3000, 'candidates': 5, 'search_metric': 'valid_f1_micro_global'}


,learning_rate,batch_size,epochs,max_length,weight_decay
0,0.00002,16,3,192,0.01
1,0.00003,16,3,192,0.01
2,0.00002,16,4,256,0.01
3,0.00001,16,4,256,0.02
4,0.00003,32,3,192,0.01


## Chargement des donnees enrichies

Le test reste issu du dataset NovelForge courant, pour garder un domaine d'evaluation comparable.

In [3]:
current = load_original_dataset(PROJECT_DIR / 'data' / 'data.csv')
mal_manga = build_enriched_dataset(PROJECT_DIR, include_original=False, include_manga=True, include_anime=False)

print(f'Dataset courant exploitable : {len(current):,}')
print(f'MAL manga exploitable : {len(mal_manga):,}')
display(summarize_labels(current).head(12))
display(summarize_labels(mal_manga).head(12))

Dataset courant exploitable : 69,588
MAL manga exploitable : 56,587


,label,count
0,Romance,29762
1,Comedy,21282
2,Drama,18702
3,Fantasy,16125
4,BL/GL Romance,14659
5,Action,12734
6,School Life,12582
7,Seinen,9235
8,Supernatural,8689
9,Shoujo,8340


,label,count
0,Adult,19214
1,Romance,14881
2,BL/GL Romance,12045
3,Comedy,11665
4,Fantasy,10987
5,Drama,9478
6,School Life,9073
7,Action,8045
8,Shoujo,7211
9,Supernatural,7035


In [4]:
current_train_full, current_temp = train_test_split(current, test_size=0.30, random_state=RANDOM_STATE, shuffle=True)
current_valid_full, current_test_full = train_test_split(current_temp, test_size=0.50, random_state=RANDOM_STATE, shuffle=True)

current_train = current_train_full.sample(n=min(CURRENT_TRAIN_ROWS, len(current_train_full)), random_state=RANDOM_STATE)
mal_train = mal_manga.sample(n=min(MAL_TRAIN_ROWS, len(mal_manga)), random_state=RANDOM_STATE)
valid_df = current_valid_full.sample(n=min(VALID_ROWS, len(current_valid_full)), random_state=RANDOM_STATE)
test_df = current_test_full.sample(n=min(TEST_ROWS, len(current_test_full)), random_state=RANDOM_STATE)

train_df = pd.concat([current_train, mal_train], ignore_index=True).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f'Train courant sample : {len(current_train):,}')
print(f'Train MAL sample : {len(mal_train):,}')
print(f'Train total : {len(train_df):,}')
print(f'Validation NovelForge : {len(valid_df):,}')
print(f'Test NovelForge : {len(test_df):,}')

Train courant sample : 8,000
Train MAL sample : 12,000
Train total : 20,000
Validation NovelForge : 2,000
Test NovelForge : 3,000


In [5]:
mlb = MultiLabelBinarizer(classes=ENRICHED_GENRE_VOCABULARY)
mlb.fit([ENRICHED_GENRE_VOCABULARY])
labels = list(mlb.classes_)
id2label = {index: label for index, label in enumerate(labels)}
label2id = {label: index for index, label in id2label.items()}

X_train = train_df['synopsis_clean'].fillna('').astype(str).to_numpy()
X_valid = valid_df['synopsis_clean'].fillna('').astype(str).to_numpy()
X_test = test_df['synopsis_clean'].fillna('').astype(str).to_numpy()

y_train = mlb.transform(train_df['genre_labels']).astype('float32')
y_valid = mlb.transform(valid_df['genre_labels']).astype('float32')
y_test = mlb.transform(test_df['genre_labels']).astype('float32')

print(f'Labels enrichis : {len(labels)}')
labels

Labels enrichis : 26


['Action',
 'Adventure',
 'Comedy',
 'Drama',
 'Fantasy',
 'Romance',
 'BL/GL Romance',
 'Adult',
 'School Life',
 'Slice of Life',
 'Supernatural',
 'Mystery',
 'Psychological',
 'Horror',
 'Historical',
 'Sci Fi',
 'Sports',
 'Martial Arts',
 'Magic',
 'Isekai',
 'Harem',
 'Mecha',
 'Seinen',
 'Shoujo',
 'Shounen',
 'Josei']

## Fonctions d'evaluation

Pour chaque candidat, on calibre un seuil global et des seuils par label sur validation, puis on compare les scores.

In [6]:
def evaluate_probabilities(y_true, probabilities, thresholds):
    y_pred = apply_thresholds(probabilities, thresholds)
    return {
        'f1_micro': f1_score(y_true, y_pred, average='micro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'jaccard_samples': jaccard_score(y_true, y_pred, average='samples', zero_division=0),
        'hamming_loss': hamming_loss(y_true, y_pred),
        'classification_report_text': classification_report(y_true, y_pred, target_names=labels, zero_division=0),
        'classification_report': classification_report(y_true, y_pred, target_names=labels, zero_division=0, output_dict=True),
    }


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Recherche d'hyperparametres

Cette cellule est la plus longue. Sur CPU, elle peut prendre plusieurs heures selon la machine. Les resultats intermediaires sont sauvegardes dans `reports/transformer_hparam_search_metrics.csv` apres chaque candidat.

In [7]:
models_dir = PROJECT_DIR / 'models'
reports_dir = PROJECT_DIR / 'reports'
search_dir = models_dir / 'transformer_hparam_search'
models_dir.mkdir(exist_ok=True)
reports_dir.mkdir(exist_ok=True)
search_dir.mkdir(exist_ok=True)

search_results = []
trained_candidates = []

for candidate_index, params in enumerate(HYPERPARAMETER_GRID, start=1):
    clear_memory()
    candidate_name = f'candidate_{candidate_index:02d}'
    output_dir = search_dir / candidate_name
    print(f'\n=== {candidate_name} / {len(HYPERPARAMETER_GRID)} ===')
    print(params)

    config = TransformerConfig(
        model_name=MODEL_NAME,
        max_length=params['max_length'],
        learning_rate=params['learning_rate'],
        train_batch_size=params['batch_size'],
        eval_batch_size=max(params['batch_size'], 16),
        epochs=params['epochs'],
        weight_decay=params['weight_decay'],
        threshold=0.5,
        output_dir=str(output_dir),
        logging_steps=50,
    )

    transformer = NovelForgeTransformer(
        num_labels=len(labels),
        id2label=id2label,
        label2id=label2id,
        config=config,
    )

    start = time.perf_counter()
    trainer = transformer.fine_tune(X_train, y_train, X_valid, y_valid)
    training_seconds = time.perf_counter() - start

    valid_proba = transformer.predict_proba(X_valid, batch_size=config.eval_batch_size)
    best_global_threshold, valid_f1_micro_global = find_best_global_threshold(y_valid, valid_proba)
    label_thresholds = find_best_label_thresholds(y_valid, valid_proba)

    valid_global_metrics = evaluate_probabilities(y_valid, valid_proba, best_global_threshold)
    valid_label_metrics = evaluate_probabilities(y_valid, valid_proba, label_thresholds)

    result = {
        'candidate': candidate_name,
        **params,
        'train_rows': len(train_df),
        'valid_rows': len(valid_df),
        'training_seconds': training_seconds,
        'best_global_threshold': best_global_threshold,
        'valid_f1_micro_global': valid_global_metrics['f1_micro'],
        'valid_f1_macro_global': valid_global_metrics['f1_macro'],
        'valid_jaccard_global': valid_global_metrics['jaccard_samples'],
        'valid_hamming_global': valid_global_metrics['hamming_loss'],
        'valid_f1_micro_per_label': valid_label_metrics['f1_micro'],
        'valid_f1_macro_per_label': valid_label_metrics['f1_macro'],
        'valid_jaccard_per_label': valid_label_metrics['jaccard_samples'],
        'valid_hamming_per_label': valid_label_metrics['hamming_loss'],
        'output_dir': str(output_dir),
    }
    search_results.append(result)
    trained_candidates.append({
        'candidate': candidate_name,
        'transformer': transformer,
        'config': config,
        'global_threshold': best_global_threshold,
        'label_thresholds': label_thresholds,
        'result': result,
    })

    results_df = pd.DataFrame(search_results).sort_values(SEARCH_METRIC, ascending=False)
    results_df.to_csv(reports_dir / 'transformer_hparam_search_metrics.csv', index=False)
    display(results_df)

results_df = pd.DataFrame(search_results).sort_values(SEARCH_METRIC, ascending=False).reset_index(drop=True)
results_df


=== candidate_01 / 5 ===
{'learning_rate': 2e-05, 'batch_size': 16, 'epochs': 3, 'max_length': 192, 'weight_decay': 0.01}


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
c:\Users\ClémentPERRET\OneDrive - EQUATERRE-VDS\Bureau\Cours\DeepLearning\venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then de

Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.246114,0.277217,0.277196,0.117459
2,0.224948,0.262967,0.345763,0.196361
3,0.211459,0.258541,0.376331,0.229790


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...


=== candidate_02 / 5 ===
{'learning_rate': 3e-05, 'batch_size': 16, 'epochs': 3, 'max_length': 192, 'weight_decay': 0.01}


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
c:\Users\ClémentPERRET\OneDrive - EQUATERRE-VDS\Bureau\Cours\DeepLearning\venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then de

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## Evaluation finale du meilleur candidat

Le meilleur candidat est selectionne sur `valid_f1_micro_global`. On l'evalue ensuite sur le test NovelForge, qui n'a pas servi a choisir les hyperparametres.

In [ ]:
best_row = results_df.iloc[0]
best_candidate_name = best_row['candidate']
best_entry = next(item for item in trained_candidates if item['candidate'] == best_candidate_name)
best_transformer = best_entry['transformer']
best_config = best_entry['config']
best_global_threshold = best_entry['global_threshold']
best_label_thresholds = best_entry['label_thresholds']

test_proba = best_transformer.predict_proba(X_test, batch_size=best_config.eval_batch_size)
test_global_metrics = evaluate_probabilities(y_test, test_proba, best_global_threshold)
test_label_metrics = evaluate_probabilities(y_test, test_proba, best_label_thresholds)

final_summary = pd.DataFrame([
    {
        'model': 'transformer_enriched_hparam_search',
        'candidate': best_candidate_name,
        'threshold_strategy': 'global',
        'train_rows': len(train_df),
        'current_train_rows': len(current_train),
        'mal_train_rows': len(mal_train),
        'valid_rows': len(valid_df),
        'test_rows': len(test_df),
        'threshold': best_global_threshold,
        **{key: test_global_metrics[key] for key in ['f1_micro', 'f1_macro', 'f1_weighted', 'jaccard_samples', 'hamming_loss']},
    },
    {
        'model': 'transformer_enriched_hparam_search',
        'candidate': best_candidate_name,
        'threshold_strategy': 'per_label',
        'train_rows': len(train_df),
        'current_train_rows': len(current_train),
        'mal_train_rows': len(mal_train),
        'valid_rows': len(valid_df),
        'test_rows': len(test_df),
        'threshold': None,
        **{key: test_label_metrics[key] for key in ['f1_micro', 'f1_macro', 'f1_weighted', 'jaccard_samples', 'hamming_loss']},
    },
])

display(results_df)
display(final_summary)

In [ ]:
report = pd.DataFrame(test_label_metrics['classification_report']).T
report.loc[labels, ['precision', 'recall', 'f1-score', 'support']].sort_values('f1-score', ascending=False)

## Sauvegarde du meilleur modele

La cellule suivante sauvegarde le meilleur candidat dans les artefacts utilises par Streamlit pour le Transformer enrichi. Elle remplace donc le Transformer enrichi precedent par la version issue de la recherche d'hyperparametres.

In [ ]:
final_output_dir = models_dir / 'transformer_enriched_novelforge'

best_transformer.save(final_output_dir)
joblib.dump(labels, models_dir / 'transformer_enriched_labels.joblib')
joblib.dump(best_label_thresholds, models_dir / 'transformer_enriched_thresholds.joblib')
joblib.dump(
    {
        'search_results': results_df,
        'summary': final_summary,
        'labels': labels,
        'best_candidate': best_candidate_name,
        'global_threshold': best_global_threshold,
        'label_thresholds': best_label_thresholds,
        'metrics_global': test_global_metrics,
        'metrics_per_label': test_label_metrics,
        'config': best_config.__dict__,
    },
    models_dir / 'transformer_enriched_metrics.joblib',
)

results_df.to_csv(reports_dir / 'transformer_hparam_search_metrics.csv', index=False)
final_summary.to_csv(reports_dir / 'transformer_enriched_metrics.csv', index=False)

print(f'Meilleur candidat : {best_candidate_name}')
print(f'Transformer enrichi optimise sauvegarde : {final_output_dir}')
print('Metriques de recherche : reports/transformer_hparam_search_metrics.csv')
print('Metriques finales : reports/transformer_enriched_metrics.csv')

## Conclusion attendue

Cette recherche permet de distinguer trois niveaux :

- l'optimisation des poids par AdamW pendant chaque entrainement ;
- la recherche d'hyperparametres entre plusieurs entrainements ;
- la calibration des seuils multilabel apres entrainement.

Si le meilleur Transformer optimise depasse la baseline enrichie du notebook 5, il devient le meilleur modele empirique du projet. Sinon, la conclusion reste utile : la baseline enrichie reste plus efficace dans les contraintes CPU, tandis que le Transformer demande plus de volume, de temps ou un GPU pour exprimer son potentiel.